In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv(r"capstone_smartgrid_20000.csv")

df.head()

,meter_id,timestamp,region,building_type,floor_area_m2,insulation_rating,hvac_age_years,solar_kw_installed,ev_charger,hour,...,consumption_kwh,next_hour_consumption_kwh,lag1_kwh,lag24_kwh,roll24_mean_kwh,roll24_std_kwh,high_usage_flag,anomaly_flag,sensor_health,outage_risk_score
0,M001,2025-01-01 00:00:00,W,commercial,113,0.709966,16.486924,2.762723,1,0,...,2.7074,2.5389,NaN,NaN,2.7074,NaN,0,0,0.973,0.1012
1,M001,2025-01-01 01:00:00,W,commercial,113,0.709966,16.486924,2.762723,1,1,...,2.5389,2.6194,2.7074,NaN,2.6232,0.1191,0,0,0.979,0.1195
2,M001,2025-01-01 02:00:00,W,commercial,113,0.709966,16.486924,2.762723,1,2,...,2.6194,2.3420,2.5389,NaN,2.6219,0.0843,0,0,0.940,0.1562
3,M001,2025-01-01 03:00:00,W,commercial,113,0.709966,16.486924,2.762723,1,3,...,2.3420,2.6295,2.6194,NaN,2.5519,0.1560,0,0,0.943,0.1309
4,M001,2025-01-01 04:00:00,W,commercial,113,0.709966,16.486924,2.762723,1,4,...,2.6295,2.2090,2.3420,NaN,2.5674,0.1394,0,0,0.995,0.1704


In [4]:
print("Dataset Shape:", df.shape)

df.info()

Dataset Shape: (20000, 31)
<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 31 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   meter_id                   20000 non-null  str    
 1   timestamp                  20000 non-null  str    
 2   region                     20000 non-null  str    
 3   building_type              20000 non-null  str    
 4   floor_area_m2              20000 non-null  int64  
 5   insulation_rating          20000 non-null  float64
 6   hvac_age_years             20000 non-null  float64
 7   solar_kw_installed         20000 non-null  float64
 8   ev_charger                 20000 non-null  int64  
 9   hour                       20000 non-null  int64  
 10  day_of_week                20000 non-null  int64  
 11  is_weekend                 20000 non-null  int64  
 12  is_holiday                 20000 non-null  int64  
 13  tariff_tier                200

In [5]:
missing_values = df.isnull().sum()

print(missing_values[missing_values > 0])

next_hour_consumption_kwh     40
lag1_kwh                      40
lag24_kwh                    960
roll24_std_kwh                40
dtype: int64


In [6]:
print("Duplicate Rows:", df.duplicated().sum())

Duplicate Rows: 0


In [7]:
df['timestamp'] = pd.to_datetime(df['timestamp'])

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 31 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   meter_id                   20000 non-null  str           
 1   timestamp                  20000 non-null  datetime64[us]
 2   region                     20000 non-null  str           
 3   building_type              20000 non-null  str           
 4   floor_area_m2              20000 non-null  int64         
 5   insulation_rating          20000 non-null  float64       
 6   hvac_age_years             20000 non-null  float64       
 7   solar_kw_installed         20000 non-null  float64       
 8   ev_charger                 20000 non-null  int64         
 9   hour                       20000 non-null  int64         
 10  day_of_week                20000 non-null  int64         
 11  is_weekend                 20000 non-null  int64         
 12  is_holiday     

In [9]:
df = df.sort_values(
    by=['meter_id', 'timestamp']
).reset_index(drop=True)

In [10]:
columns_to_fill = [
    'next_hour_consumption_kwh',
    'lag1_kwh',
    'lag24_kwh',
    'roll24_std_kwh'
]

df[columns_to_fill] = (
    df.groupby('meter_id')[columns_to_fill]
      .transform(lambda x: x.interpolate(limit_direction='both'))
)

In [11]:
df.isnull().sum()

meter_id                     0
timestamp                    0
region                       0
building_type                0
floor_area_m2                0
insulation_rating            0
hvac_age_years               0
solar_kw_installed           0
ev_charger                   0
hour                         0
day_of_week                  0
is_weekend                   0
is_holiday                   0
tariff_tier                  0
temp_c                       0
humidity_pct                 0
wind_ms                      0
precip_mm                    0
grid_price_usd_per_kwh       0
occupancy_index              0
solar_generation_kwh         0
consumption_kwh              0
next_hour_consumption_kwh    0
lag1_kwh                     0
lag24_kwh                    0
roll24_mean_kwh              0
roll24_std_kwh               0
high_usage_flag              0
anomaly_flag                 0
sensor_health                0
outage_risk_score            0
dtype: int64

In [12]:
df = df.drop_duplicates()

In [13]:
energy_columns = [
    'solar_generation_kwh',
    'consumption_kwh',
    'next_hour_consumption_kwh'
]

for col in energy_columns:
    print(col, df[col].min())

solar_generation_kwh 0.0
consumption_kwh 0.0
next_hour_consumption_kwh 0.0


In [14]:
df['net_energy_kwh'] = (
    df['consumption_kwh']
    - df['solar_generation_kwh']
)


In [15]:
df['date'] = df['timestamp'].dt.date

In [16]:
df['month'] = df['timestamp'].dt.month

In [17]:
df['day_name'] = df['timestamp'].dt.day_name()

In [18]:
df['peak_period'] = np.where(
    df['hour'].between(18, 22),
    'Peak Hours',
    'Normal Hours'
)

In [19]:
df['anomaly_status'] = np.where(
    df['anomaly_flag'] == 1,
    'Anomaly Detected',
    'Normal'
)

In [20]:
def risk_category(score):

    if score < 0.3:
        return 'Low Risk'

    elif score < 0.7:
        return 'Medium Risk'

    else:
        return 'High Risk'


df['risk_category'] = df['outage_risk_score'].apply(
    risk_category
)

In [21]:
df.to_csv(
    "smartgrid_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!
